In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/cnrieiit/mqttset/requirements.txt
/kaggle/input/datasets/cnrieiit/mqttset/code/processingdata_ml.py
/kaggle/input/datasets/cnrieiit/mqttset/code/parser/rawparsing_normal.py
/kaggle/input/datasets/cnrieiit/mqttset/code/parser/rawparsing_augmented.py
/kaggle/input/datasets/cnrieiit/mqttset/Data/README.md
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/malformed.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/flood.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/bruteforce.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/malaria.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/legitimate_1w.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/slowite.csv
/kaggle/input/datasets/cnrieiit/mqttset/Data/PCAP/bruteforce.pcapng
/kaggle/input/datasets/cnrieiit/mqttset/Data/PCAP/capture_1w.pcap
/kaggle/input/datasets/cnrieiit/mqttset/Data/PCAP/slowite.pcap
/kaggle/input/datasets/cnrieiit/mqttset/Data/PCAP/malformed.pcap
/kaggle/input/datasets/cnrieiit/mqtt

In [2]:
import pandas as pd
import numpy as np

# ============================
# NORMAL TRAFFIC
# ============================

normal = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/legitimate_1w.csv'
)

# ============================
# ATTACKS
# ============================

flood = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/flood.csv'
)

bruteforce = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/bruteforce.csv'
)

malaria = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/malaria.csv'
)

malformed = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/malformed.csv'
)

slowite = pd.read_csv(
'/kaggle/input/datasets/cnrieiit/mqttset/Data/CSV/slowite.csv'
)

print("Normal:", normal.shape)
print("Flood:", flood.shape)
print("Bruteforce:", bruteforce.shape)
print("Malaria:", malaria.shape)
print("Malformed:", malformed.shape)
print("Slowite:", slowite.shape)

/tmp/ipykernel_24/2400055045.py:8: DtypeWarning: Columns (21,23,34,39,45) have mixed types. Specify dtype option on import or set low_memory=False.
  normal = pd.read_csv(


Normal: (11915716, 60)
Flood: (613, 60)
Bruteforce: (14501, 60)
Malaria: (130223, 60)
Malformed: (10924, 60)
Slowite: (9202, 60)


/tmp/ipykernel_24/2400055045.py:24: DtypeWarning: Columns (21,23,34,45) have mixed types. Specify dtype option on import or set low_memory=False.
  malaria = pd.read_csv(


In [3]:
print(normal['mqtt.msgtype'].value_counts())

mqtt.msgtype
3.0     7307797
12.0     100850
13.0     100850
1.0          10
2.0          10
8.0           7
9.0           7
Name: count, dtype: int64


In [4]:
attacks = {

    "Flood": flood,
    "Bruteforce": bruteforce,
    "Malaria": malaria,
    "Malformed": malformed,
    "Slowite": slowite
}

for name, df in attacks.items():

    print("\n======================")
    print(name)
    print("======================")

    print(
        df['mqtt.msgtype'].value_counts()
    )


Flood
mqtt.msgtype
3.0     295
13.0      3
12.0      3
1.0       1
2.0       1
Name: count, dtype: int64

Bruteforce
mqtt.msgtype
1.0     1430
2.0     1430
3.0       55
14.0       2
8.0        2
9.0        2
Name: count, dtype: int64

Malaria
mqtt.msgtype
3.0     54431
4.0     38841
1.0        15
2.0        15
14.0       15
Name: count, dtype: int64

Malformed
mqtt.msgtype
3.0     1777
1.0      820
2.0      820
8.0       80
9.0       80
4.0       39
5.0       39
14.0       1
Name: count, dtype: int64

Slowite
mqtt.msgtype
1.0    1016
2.0    1016
Name: count, dtype: int64


In [5]:
import numpy as np
from scipy.spatial.distance import euclidean

# ==================================================
# PROBABILITY DISTRIBUTION
# ==================================================

def get_distribution(df):

    counts = df['mqtt.msgtype'].value_counts()

    probs = counts / counts.sum()

    return probs


# ==================================================
# HELLINGER DISTANCE
# ==================================================

def hellinger(p, q):

    keys = sorted(
        list(set(p.index).union(set(q.index)))
    )

    p_vec = np.array(
        [p.get(k, 0) for k in keys]
    )

    q_vec = np.array(
        [q.get(k, 0) for k in keys]
    )

    return euclidean(
        np.sqrt(p_vec),
        np.sqrt(q_vec)
    ) / np.sqrt(2)

In [6]:
normal_profile = get_distribution(normal)

results = []

for name, df in attacks.items():

    attack_profile = get_distribution(df)

    hd = hellinger(
        normal_profile,
        attack_profile
    )

    results.append(
        [name, hd]
    )

    print(
        f"{name} --> HD = {hd:.6f}"
    )

Flood --> HD = 0.058638
Bruteforce --> HD = 0.928962
Malaria --> HD = 0.496552
Malformed --> HD = 0.557563
Slowite --> HD = 0.999184


In [7]:
normal_profile = get_distribution(normal)

flood_profile = get_distribution(flood)

comparison = pd.DataFrame({

    'Normal': normal_profile,

    'Flood': flood_profile

}).fillna(0)

print(comparison)

                    Normal     Flood
mqtt.msgtype                        
1.0           1.331641e-06  0.003300
2.0           1.331641e-06  0.003300
3.0           9.731363e-01  0.973597
8.0           9.321488e-07  0.000000
9.0           9.321488e-07  0.000000
12.0          1.342960e-02  0.009901
13.0          1.342960e-02  0.009901


In [8]:
features = [

    'mqtt.topic_len',

    'mqtt.len',

    'mqtt.kalive',

    'mqtt.username_len',

    'mqtt.passwd_len'
]

for f in features:

    print("\n", f)

    print("Normal")

    print(normal[f].describe())

    print("\nFlood")

    print(flood[f].describe())


 mqtt.topic_len
Normal
count    7.307804e+06
mean     7.342645e+00
std      1.407033e+00
min      3.000000e+00
25%      6.000000e+00
50%      8.000000e+00
75%      8.000000e+00
max      1.500000e+01
Name: mqtt.topic_len, dtype: float64

Flood
count    298.0
mean      11.0
std        0.0
min       11.0
25%       11.0
50%       11.0
75%       11.0
max       11.0
Name: mqtt.topic_len, dtype: float64

 mqtt.len
Normal
count    7.509531e+06
mean     1.035092e+01
std      2.143442e+00
min      0.000000e+00
25%      1.100000e+01
50%      1.100000e+01
75%      1.100000e+01
max      3.500000e+01
Name: mqtt.len, dtype: float64

Flood
count    303.000000
mean      15.155116
std        2.617889
min        0.000000
25%       15.000000
50%       15.000000
75%       16.000000
max       35.000000
Name: mqtt.len, dtype: float64

 mqtt.kalive
Normal
count    10.0
mean     60.0
std       0.0
min      60.0
25%      60.0
50%      60.0
75%      60.0
max      60.0
Name: mqtt.kalive, dtype: float64

Flood
co

In [9]:
import numpy as np

def get_numeric_distribution(series, bins=10):

    series = series.dropna()

    # Empty feature
    if len(series) == 0:

        return np.ones(bins) / bins

    hist, _ = np.histogram(

        series,

        bins=bins
    )

    hist = hist.astype(float)

    total = hist.sum()

    if total == 0:

        return np.ones(bins) / bins

    return hist / total

In [10]:
from scipy.spatial.distance import euclidean

def hellinger_hist(p, q):

    return euclidean(

        np.sqrt(p),

        np.sqrt(q)

    ) / np.sqrt(2)

In [11]:
# ==========================================
# NORMAL PROFILES
# ==========================================

normal_msgtype = get_distribution(
    normal
)

normal_topic = get_numeric_distribution(
    normal['mqtt.topic_len']
)

normal_msglen = get_numeric_distribution(
    normal['mqtt.len']
)

In [12]:
for name, df in attacks.items():

    print("\n===================")
    print(name)
    print("===================")

    print("topic_len non-null:",
          df['mqtt.topic_len'].notna().sum())

    print("mqtt.len non-null:",
          df['mqtt.len'].notna().sum())


Flood
topic_len non-null: 298
mqtt.len non-null: 303

Bruteforce
topic_len non-null: 57
mqtt.len non-null: 2921

Malaria
topic_len non-null: 54431
mqtt.len non-null: 93317

Malformed
topic_len non-null: 1857
mqtt.len non-null: 3656

Slowite
topic_len non-null: 0
mqtt.len non-null: 2032


In [13]:
attacks = {

    "Flood": flood,
    "Bruteforce": bruteforce,
    "Malaria": malaria,
    "Malformed": malformed,
    "Slowite": slowite
}

results_B = []

for name, df in attacks.items():

    # =====================================
    # MESSAGE TYPE HD
    # =====================================

    attack_msgtype = get_distribution(df)

    hd_msgtype = hellinger(

        normal_msgtype,

        attack_msgtype
    )

    # =====================================
    # TOPIC LENGTH HD
    # =====================================

    attack_topic = get_numeric_distribution(

        df['mqtt.topic_len']
    )

    hd_topic = hellinger_hist(

        normal_topic,

        attack_topic
    )

    # =====================================
    # MESSAGE LENGTH HD
    # =====================================

    attack_msglen = get_numeric_distribution(

        df['mqtt.len']
    )

    hd_msglen = hellinger_hist(

        normal_msglen,

        attack_msglen
    )

    # =====================================
    # COMBINED HD
    # =====================================

    hd_combined = (

        hd_msgtype +

        hd_topic +

        hd_msglen

    ) / 3

    results_B.append([

        name,

        hd_msgtype,

        hd_topic,

        hd_msglen,

        hd_combined
    ])

    print(
        f"{name}: "
        f"Combined HD = {hd_combined:.4f}"
    )

Flood: Combined HD = 0.5989
Bruteforce: Combined HD = 0.8516
Malaria: Combined HD = 0.7757
Malformed: Combined HD = 0.7142
Slowite: Combined HD = 0.8455


In [14]:
print(hd_msgtype)
print(hd_topic)
print(hd_msglen)

0.9991836885448567
0.5976005126594113
0.9398366803469682
